# Facial recognition
will create unique id for every face on will output all the photos the person is in

### Imprting everything needed

In [12]:
import pandas as pd 
import numpy as np
import cv2 
import os
import chromadb
from insightface.app import FaceAnalysis

In [13]:
model = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider'])
model.prepare(ctx_id=0, det_size=(640, 640))

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/arc/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], wi

In [14]:
chroma_client = chromadb.PersistentClient(path = "sdf")
collection = chroma_client.get_or_create_collection(name="face_embeddings",
                                                    metadata={"hnsw:space": "cosine"})

In [15]:
import uuid
import shutil

### Creating a filter for bluriness

In [16]:
def quality(img,face,blur,conf):
    if face.det_score < conf:
        return False
    bbox = face.bbox.astype(int)
    x1, y1, x2, y2 = bbox
    y1,y2 = max(0,y1), min(img.shape[0],y2)
    x1,x2 = max(0,x1), min(img.shape[1],x2)
    face_crop = img[y1:y2, x1:x2]
    if face_crop.size == 0:
        return False
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < blur:
        return False
    return True

### directories and stuff

In [17]:
dump = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb"
output = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/facial_output"
os.makedirs(output, exist_ok=True)
discard_fol = os.path.join(output, "_Discar")
os.makedirs(discard_fol, exist_ok=True)

Parameters(will make it ui centric later)

In [21]:
dis_thres = 0.5
blur = 80
conf = 0.6

Generating embedding

In [22]:
for filename in os.listdir(dump):
    folder_path = os.path.join(dump, filename)
    if not os.path.isdir(folder_path):
        continue
    for img_path in os.listdir(folder_path):
        filepath = os.path.join(folder_path,img_path)
        img = cv2.imread(filepath)
        faces = model.get(img)
        for face in faces:
            Blur = quality(img,face,blur,conf)
            if not Blur:
                print(f"skipped {filepath} due to blurr")
                continue
            embedding = face.embedding.tolist()
            results = collection.query(query_embeddings=[embedding], n_results=1)
            per_id = None
            if results['distances'][0] and results['distances'][0][0] < dis_thres:
                per_id = results['metadatas'][0][0]['per_id']
            else:
                per_id = f"Person_{str(uuid.uuid4())[:8]}"
                face_vector_id = str(uuid.uuid4())
                
                collection.add(
                    ids=[face_vector_id], 
                    embeddings=[embedding], 
                    metadatas=[{"per_id": per_id, "original_folder": folder_path,"filename": filename}]
                    )
            print(f"Image: {filepath}, Assigned ID: {per_id}")
            perid_out_dir = os.path.join(output,per_id)
            os.makedirs(perid_out_dir,exist_ok=True)
            shutil.copy(filepath,os.path.join(perid_out_dir,img_path))

Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb/madonna/httpecximagesamazoncomimagesIfmaBKWLACULSRjpg.jpg, Assigned ID: Person_4ac70b95
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb/madonna/httpssmediacacheakpinimgcomxffabffabbbcfbceaedjpg.jpg, Assigned ID: Person_bae3a02d
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb/madonna/httpimagegaladevcmseamadonnaprivatdetektivsquaretopsquarejpgv.jpg, Assigned ID: Person_2c215d4c
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb/madonna/httpwwwblackdogfilmscomwordpresswpcontentuploadsmadonnacelebrationxjpg.jpg, Assigned ID: Person_4292c496
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/f_celeb/madonna/httpsuploadwikimediaorgwikipediacommonsthumbaaMadonnaatthepremiereofIAmBecauseWeArejpgpxMadonnaatthepremiereofIAmBecauseWeArejpg.jpg, Assigned ID: Person_c3dc8364
Image: /mnt/

In [20]:
import onnxruntime as ort
print("Available Providers:", ort.get_available_providers())
print(ort.__version__)   

Available Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
1.24.3
